# Runing a simple RAG using OGX and OLLAMA using OpenAI

In [1]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8321/v1", api_key="fake")

# Upload our documents
files = [
    "kubernetes_doc.txt",
    "ogx-ollama_doc.txt",
    "my-resume_doc.txt",
    "event-agenda-doc.txt",
    "training-courses-doc.txt"
]

file_ids = []

for filename in files:
    uploaded_file = client.files.create(
        file=open(filename, "rb"),
        purpose="assistants",
    )
    file_ids.append(uploaded_file.id)
    print(f"Uploaded {filename}: {uploaded_file.id}")

    vector_store = client.vector_stores.create(
    name="my-docs",
    file_ids=file_ids,
)

print(f"Vector Store ID: {vector_store.id}")


Uploaded kubernetes_doc.txt: file-e089c692db434b3d9c6d9f4f816f7ed5
Uploaded ogx-ollama_doc.txt: file-00799b5d951e4f93bb33c5514cc843ae
Uploaded my-resume_doc.txt: file-e716e01b7f9144b8b4d7d2ae425cbe6c
Uploaded event-agenda-doc.txt: file-b76cc05129d84bfc93ac069f2f516f01
Uploaded training-courses-doc.txt: file-ac6d010933ad403daecf5e06ae10d21e
Vector Store ID: vs_a6835986-d735-4b3c-a529-b3172715066f


Helper Functions

In [2]:
model = "ollama/llama3.2:3b"
relevant_threshold = 0.002

def get_top_score(output_list):
    top_score = 0.0

    for item in output_list:
        if getattr(item, "type", None) == "file_search_call":
            for r in getattr(item, "results", []):
                score = getattr(r, "score", 0.0) or 0.0
                top_score = max(top_score, score)

    return top_score

def not_relevant(top_score):
    if top_score < relevant_threshold:
        print("Didn't find any relevant answer...")
        return True
    else:
        print("Found relevant answer...")
        return False

def normal_llm(query: str):
    response = client.responses.create(
        model=model,
        temperature=0.0,
        input=query
    )

    return response
    
def ask_rag(query: str, full_response: bool = False):
    response = client.responses.create(
        model=model,
        temperature=0.0,
        input=query,
        tools=[
            {
                "type": "file_search",
                "vector_store_ids": [vector_store.id],
                "max_num_results": 1,
            }
        ],
    )
    top_score = get_top_score(response.output)
    print("top_score=", top_score)
    if not_relevant(top_score):
        # revert back to normal LLM query
        response = normal_llm(query);
    #format the response
    if full_response:
        return response
    return response.output_text

Test Cases

In [3]:
print(ask_rag("who is Osama Oransa according to the documents", False))

top_score= 0.0044302822391826474
Found relevant answer...
Osama Oransa is an SSA (Specialist Solution Architect) in the MENA region at Red Hat, based in Dubai.


In [4]:
print(ask_rag("What does the document say about Osa Ora?",False))

top_score= 0.002887603335936183
Found relevant answer...
I couldn't find any information on "Osa Ora". However, I found a mention of "Osama Oransa" which is an SSA (Specialist Solution Architect) in the MENA region at Red Hat based in Dubai. He enjoys playing tennis.


In [5]:
print(ask_rag("What does the document say about global warming?", False))

top_score= 0.0017191704079399146
Didn't find any relevant answer...
I don't have any information about a specific document. If you could provide more context or details about the document, I'd be happy to try and help you understand what it says about global warming.

If you're looking for general information on global warming, I can provide some information. Global warming refers to the long-term rise in the overall temperature of the Earth's atmosphere, primarily caused by human activities such as burning fossil fuels and deforestation.

The Intergovernmental Panel on Climate Change (IPCC) is a leading international organization that provides scientific advice on climate change. According to the IPCC, global warming is real, and it's primarily caused by human activities that release greenhouse gases, such as carbon dioxide and methane, into the atmosphere.

The document may discuss various aspects of global warming, including:

1. Causes: Human activities such as burning fossil fuels

In [6]:
print(ask_rag("What does the document say about OGX?"))

top_score= 0.003523800818363166
Found relevant answer...
OGX is an AI gateway that exposes OpenAI-compatible APIs. It can connect to Ollama models and provide file search capabilities. Ollama is a local model runner that executes LLMs like Llama and DeepSeek.


In [7]:
print(ask_rag("What does the document say about sessions delivered by Osama Oransa in Cairo DevDay and duration of each session",False))

top_score= 0.004294598894450088
Found relevant answer...
Osama Oransa delivered two sessions at Cairo DevDay. The first session, titled "Llama Stack / OGX: One AI API to Rule Them All", lasted from 11:10-11:40. The second session, titled "Hands On Lab", took place from 15:40-16:10.


In [8]:
print(ask_rag("What does the document say about sessions delivered by Mourad in Cairo DevDay",False))

top_score= 0.004460541367444242
Found relevant answer...
Mourad Ali delivered two sessions at Cairo DevDay: "OpenShift AI" and "Scaling Application Modernization with AI".


In [9]:
print(ask_rag("What does the document say about the total session times for Osama Oransa sessions in Cairo DevDay",False))

top_score= 0.004308289908151698
Found relevant answer...
The total session time for Osama Oransa sessions in Cairo DevDay was 1 hour and 40 minutes (100 minutes). This information can be found on page 12 of the event agenda document, where it is listed under the "Hands On Lab" section.


In [10]:
print(ask_rag("What does the document say about course Course: AI-101?",False))

top_score= 0.004310330652925383
Found relevant answer...
The document provides information on various courses offered by ACME Corp., including Course: AI-101. This course is titled "Introduction to Enterprise AI" and has a duration of 2 days. It is instructed by Ahmed Hassan and requires basic IT knowledge as a prerequisite. The course covers topics such as fundamentals of generative AI, large language models, prompt engineering, enterprise AI use cases, and responsible AI.

Additionally, the document mentions two other courses: Course: AI-201 and Course: AI-301. Course: AI-201 is titled "Building RAG Applications" and has a duration of 3 days. It is instructed by Sarah Ali and requires Python fundamentals as a prerequisite. Course: AI-301 is titled "Agentic AI and MCP Integration" and also has a duration of 3 days. It is instructed by Osama Oransa and requires completion of Course: AI-201 as a prerequisite.

Lastly, the document mentions Course: AI-401, which is titled "Enterprise AI 

In [11]:
print(ask_rag("who is the Course: AI-201 instructor?",False))

top_score= 0.0040393593458723015
Found relevant answer...
The instructor for Course: AI-201 is Sarah Ali. She teaches Building RAG Applications over a duration of 3 days with no certification required.


In [12]:
print(ask_rag("How many days i need to finish both course AI-201 and AI-401 as per the document?",False))

top_score= 0.003440127627391176
Found relevant answer...
Based on the document, to finish both courses AI-201 and AI-401, you would need 8 days (3 days for AI-201 + 5 days for AI-401).


In [13]:
print(ask_rag("which courses has a Certification as per the document?")) 

top_score= 0.0024911621825037824
Found relevant answer...
Based on the search results, the courses with certification are:

* AI-101: Introduction to Enterprise AI
* AI-301: Agentic AI and MCP Integration
* AI-401: Enterprise AI on OpenShift


In [14]:
#clean up
client.vector_stores.delete(vector_store.id)

VectorStoreDeleted(id='vs_a6835986-d735-4b3c-a529-b3172715066f', deleted=True, object='vector_store.deleted')